# 01. Audit data dan eksplorasi (Bab 5.1)
Notebook ini memanggil `src/preprocessing.py` dan membaca `results/audit_summary.json` (dibuat oleh `src/audit_data.py`). Tidak ada model yang dilatih. Kelas positif = phishing (`label == 0`, lihat `docs/decisions.md` D1).

In [1]:
import os, sys
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
sys.path.insert(0, 'src')
import json
import pandas as pd
pd.set_option('display.width', 160)
print('akar repositori:', Path.cwd().name)

akar repositori: ttki-phishing-detection


## Ringkasan audit (dari `src/audit_data.py`)

In [2]:
a = json.load(open('results/audit_summary.json', encoding='utf-8'))
for k in ['baris','kolom','missing_total','inf_total','label_asli','url_duplikat_baris','url_unik','host_unik','etld1_unik','kelompok_fitur']:
    print(f'{k:24s}', a[k])

baris                    235795
kolom                    56
missing_total            0
inf_total                0
label_asli               {'1_legitimate': 134850, '0_phishing': 100945}
url_duplikat_baris       850
url_unik                 235370
host_unik                220086
etld1_unik               175353
kelompok_fitur           {'leksikal_url': 18, 'turunan_korpus': 4, 'konten_html': 28}


## Data setelah pembersihan (duplikat URL dibuang, kelas positif phishing, grup eTLD+1)

In [3]:
from preprocessing import muat_bersih
df = muat_bersih()
print('baris:', len(df), '| proporsi phishing:', round(df['phish'].mean(), 4), '| grup eTLD+1:', df['grup'].nunique())
df['phish'].map({1:'phishing', 0:'legitimate'}).value_counts()

baris: 235370 | proporsi phishing: 0.4271 | grup eTLD+1: 175509


phish
legitimate    134850
phishing      100520
Name: count, dtype: int64

## Artefak bentuk URL
Seluruh URL legitimate berbentuk `https://www.<domain>` tanpa path, sedangkan hampir semua URL phishing tidak (analisis deskriptif pada seluruh data; lihat `docs/decisions.md` Perubahan 3).

In [4]:
p = df['URL'].str.replace(r'^https?://', '', regex=True)
bentuk = (~p.str.contains(r'/.+|\?', regex=True)) & p.str.startswith('www.') & df['URL'].str.startswith('https')
print(pd.crosstab(df['phish'].map({1:'phishing',0:'legitimate'}), bentuk.map({True:'https://www.<domain>, tanpa path', False:'bentuk lain'})))

URL         bentuk lain  https://www.<domain>, tanpa path
phish                                                    
legitimate            0                            134850
phishing          99484                              1036


## AUC satu fitur (>= 0,95) dan kelompok fitur

In [5]:
print(pd.Series(a['auc_satu_fitur_di_atas_0_95']).to_string())
print('AUC satu fitur leksikal tertinggi:', a['auc_satu_fitur_leksikal_maks'])

URLSimilarityIndex    0.9961
LineOfCode            0.9903
NoOfExternalRef       0.9880
NoOfImage             0.9796
NoOfSelfRef           0.9725
NoOfJS                0.9708
NoOfCSS               0.9576
AUC satu fitur leksikal tertinggi: 0.8238
